In [ ]:
# ============================================================
# 1. УСТАНОВКА ЗАВИСИМОСТЕЙ И ИМПОРТ
# ============================================================

!pip install -q ultralytics
!pip install -q "polars>=1.28,<1.33"

import os
import json
import random
from pathlib import Path
from collections import defaultdict

import yaml
import pandas as pd
from PIL import Image
from ultralytics import YOLO

In [ ]:
# ============================================================
# FIX: Патч read_results_csv — polars → pandas
# ============================================================

from ultralytics.engine.trainer import BaseTrainer

def _read_results_csv_pandas(self):
    csv_path = self.csv
    if csv_path.exists() and csv_path.stat().st_size > 0:
        try:
            df = pd.read_csv(csv_path)
            df.columns = [c.strip() for c in df.columns]
            return {col: df[col].tolist() for col in df.columns}
        except Exception:
            return {}
    return {}

BaseTrainer.read_results_csv = _read_results_csv_pandas

In [ ]:
# ============================================================
# 2. НАСТРОЙКА ПУТЕЙ
# ============================================================

BASE_PATH   = Path("/kaggle/input/competitions/dl-lab-2-stuff-detection")
DATASET_SRC = BASE_PATH / "yolo_dataset" / "yolo_dataset"
TEST_IMAGES = BASE_PATH / "test_images"  / "test_images"
SAMPLE_SUB  = BASE_PATH / "sample_sub.csv"

WORK        = Path("/kaggle/working")
DATASET_DIR = WORK / "dataset"
RUNS_DIR    = WORK / "runs"

In [ ]:
# ============================================================
# 3. ПОДСЧЁТ СОТРУДНИКОВ В LABEL-ФАЙЛЕ
# ============================================================

STAFF_CLASS = 1

def count_staff(label_path):
    """Вернуть число объектов класса STAFF_CLASS в label-файле."""
    if not label_path.exists():
        return 0
    text = label_path.read_text().strip()
    if not text:
        return 0
    return sum(
        1 for line in text.splitlines()
        if len(line.split()) >= 5 and int(line.split()[0]) == STAFF_CLASS
    )

In [ ]:
# ============================================================
# 4. СТРАТИФИЦИРОВАННЫЙ TRAIN / VAL SPLIT
#    Группируем изображения по кол-ву сотрудников (0, 1, 2, …).
#    Из каждой группы 15 % уходит в val → пропорции сохранены.
# ============================================================

def create_stratified_split(source_root, dest_root,
                            val_ratio=0.15, seed=42):
    src_imgs = source_root / "train" / "images"
    src_lbls = source_root / "train" / "labels"

    all_images = sorted(
        f for f in src_imgs.iterdir()
        if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
    )

    # ---------- группировка ----------
    groups = defaultdict(list)
    for img in all_images:
        n = count_staff(src_lbls / f"{img.stem}.txt")
        groups[n].append(img)

    # ---------- стратифицированное разбиение ----------
    random.seed(seed)
    train_list, val_list = [], []

    # Для подробной статистики
    train_groups = defaultdict(int)
    val_groups = defaultdict(int)

    for k in sorted(groups):
        imgs = groups[k][:]
        random.shuffle(imgs)
        if len(imgs) <= 1:               # слишком мало — всё в train
            train_list.extend(imgs)
            train_groups[k] += len(imgs)
            continue
        n_val = max(1, int(len(imgs) * val_ratio))
        val_list.extend(imgs[:n_val])
        train_list.extend(imgs[n_val:])
        val_groups[k] += n_val
        train_groups[k] += len(imgs) - n_val

    # ---------- symlinks ----------
    for split, files in [("train", train_list), ("val", val_list)]:
        img_d = dest_root / split / "images"
        lbl_d = dest_root / split / "labels"
        img_d.mkdir(parents=True, exist_ok=True)
        lbl_d.mkdir(parents=True, exist_ok=True)

        for img in files:
            dst = img_d / img.name
            if not dst.exists():
                os.symlink(str(img.resolve()), str(dst))
            lbl = src_lbls / f"{img.stem}.txt"
            if lbl.exists():
                dl = lbl_d / lbl.name
                if not dl.exists():
                    os.symlink(str(lbl.resolve()), str(dl))

    # ---------- подробный вывод ----------
    all_keys = sorted(set(list(train_groups.keys()) + list(val_groups.keys())))

    print("=" * 60)
    print("РЕЗУЛЬТАТ СТРАТИФИЦИРОВАННОГО SPLIT")
    print("=" * 60)
    print(f"  Всего изображений:  {len(all_images)}")
    print(f"  Train изображений:  {len(train_list)}")
    print(f"  Val   изображений:  {len(val_list)}")
    print()
    print(f"  {'Кол-во staff':<16} {'Train':>8} {'Val':>8} {'Всего':>8} {'Val %':>8}")
    print(f"  {'—' * 52}")
    for k in all_keys:
        t = train_groups[k]
        v = val_groups[k]
        total = t + v
        pct = f"{v / total * 100:.1f}%" if total > 0 else "—"
        print(f"  {k:<16} {t:>8} {v:>8} {total:>8} {pct:>8}")
    print(f"  {'—' * 52}")
    print(f"  {'ИТОГО':<16} {len(train_list):>8} {len(val_list):>8} "
          f"{len(all_images):>8} "
          f"{round(len(val_list)/len(all_images)*100, 1):>7}%")
    print("=" * 60)

    return train_list

In [ ]:
train_images = create_stratified_split(
    DATASET_SRC, DATASET_DIR, val_ratio=0.15, seed=42
)

In [ ]:
# ============================================================
# 5. НАРЕЗКА ТАЙЛОВ (×4) ДЛЯ TRAIN
#
#    1280×720 → 4 тайла 640×360 с ~10 % перекрытием.
#    Тайлы добавляются к оригиналам → train ×5.
#    Для каждого тайла labels пересчитываются:
#      • bbox обрезается по границам тайла;
#      • если видимая доля < min_vis — пропускается.
# ============================================================

def create_tiles(train_images, source_root, dest_root,
                 img_w=1280, img_h=720,
                 tile_w=640, tile_h=360,
                 overlap_frac=0.10,
                 min_vis=0.30):

    src_lbls = source_root / "train" / "labels"
    dst_imgs = dest_root / "train" / "images"
    dst_lbls = dest_root / "train" / "labels"

    ov_x = int(tile_w * overlap_frac)          # 64 px
    ov_y = int(tile_h * overlap_frac)          # 36 px

    # Начальные координаты каждого тайла (x0, y0) и суффикс имени
    origins = [
        (0,              0,             "tl"),   # top-left
        (tile_w - ov_x,  0,             "tr"),   # top-right  (576, 0)
        (0,              tile_h - ov_y, "bl"),   # bottom-left (0, 324)
        (tile_w - ov_x,  tile_h - ov_y, "br"),  # bottom-right (576, 324)
    ]

    n_tiles = 0
    n_orig = len(train_images)

    for i, img_path in enumerate(train_images):

        pil = Image.open(str(img_path))
        W, H = pil.size                        # 1280, 720

        # ---- чтение labels → абсолютные пиксели ----
        lbl_path = src_lbls / f"{img_path.stem}.txt"
        boxes = []
        if lbl_path.exists():
            text = lbl_path.read_text().strip()
            if text:
                for ln in text.splitlines():
                    p = ln.split()
                    if len(p) < 5:
                        continue
                    c  = int(p[0])
                    cx = float(p[1]) * W
                    cy = float(p[2]) * H
                    bw = float(p[3]) * W
                    bh = float(p[4]) * H
                    boxes.append((c, cx, cy, bw, bh))

        # ---- для каждого тайла ----
        for x0, y0, sfx in origins:
            xe = min(x0 + tile_w, W)
            ye = min(y0 + tile_h, H)
            tw = xe - x0
            th = ye - y0

            crop = pil.crop((x0, y0, xe, ye))

            # пересчёт labels
            lines = []
            for c, cx, cy, bw, bh in boxes:
                bx1, by1 = cx - bw / 2, cy - bh / 2
                bx2, by2 = cx + bw / 2, cy + bh / 2

                # clip по тайлу
                cx1 = max(bx1, x0);  cy1 = max(by1, y0)
                cx2 = min(bx2, xe);  cy2 = min(by2, ye)

                cw_ = cx2 - cx1
                ch_ = cy2 - cy1
                if cw_ <= 2 or ch_ <= 2:
                    continue
                if bw * bh > 0 and (cw_ * ch_) / (bw * bh) < min_vis:
                    continue

                # YOLO-формат относительно тайла
                ncx = max(0, min(1, ((cx1 + cx2) / 2 - x0) / tw))
                ncy = max(0, min(1, ((cy1 + cy2) / 2 - y0) / th))
                nw  = max(0.001, min(1, cw_ / tw))
                nh  = max(0.001, min(1, ch_ / th))

                lines.append(
                    f"{c} {ncx:.6f} {ncy:.6f} {nw:.6f} {nh:.6f}"
                )

            name = f"{img_path.stem}_{sfx}"
            crop.save(str(dst_imgs / f"{name}.jpg"), quality=95)
            (dst_lbls / f"{name}.txt").write_text("\n".join(lines))

            n_tiles += 1

        pil.close()

    total = len([f for f in dst_imgs.iterdir()
                 if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])

    print()
    print("=" * 60)
    print("РЕЗУЛЬТАТ ТАЙЛИНГА")
    print("=" * 60)
    print(f"  Оригиналов в train (до тайлинга):   {n_orig}")
    print(f"  Тайлов создано:                     {n_tiles}")
    print(f"  Всего в train (оригиналы + тайлы):  {total}")
    print(f"  Увеличение:                         ×{total / n_orig:.1f}")
    print("=" * 60)

In [ ]:
create_tiles(train_images, DATASET_SRC, DATASET_DIR)

In [ ]:
# ============================================================
# 6. ИСПРАВЛЕНИЕ data.yaml
# ============================================================

def fix_data_yaml(source_dir, new_root, output):
    for c in [source_dir / "train" / "data.yaml",
              source_dir / "data.yaml"]:
        if c.exists():
            orig = c
            break
    else:
        raise FileNotFoundError("data.yaml не найден")

    with open(orig) as f:
        cfg = yaml.safe_load(f)

    cfg["path"] = str(new_root)
    cfg["val"]  = "val/images"

    with open(output, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
    print(f"data.yaml → {output}")
    return str(output)


yaml_path = fix_data_yaml(DATASET_SRC, DATASET_DIR, WORK / "data.yaml")

In [ ]:
# ============================================================
# 7. ОБУЧЕНИЕ МОДЕЛИ  (YOLO11m, imgsz=640)
# ============================================================

def train_model(data_yaml, model_name,
                epochs, imgsz, batch,
                project="runs", run_name="train", seed=42):

    model = YOLO(model_name)

    model.train(
        data=data_yaml,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        project=project,
        name=run_name,
        exist_ok=True,

        device=0,        # используем только первый T4
        amp=True,        # Mixed Precision — максимум от Tensor Cores

        # ── Сохранение и ранняя остановка ──
        save=True,
        save_period=5,
        patience=10,

        # ── Оптимизатор ──
        optimizer="AdamW",
        lr0=0.0005,
        lrf=0.01,
        weight_decay=0.0005,
        warmup_epochs=3,
        cos_lr=True,

        # ── Веса loss ──
        box=7.5,
        cls=1.5,
        dfl=1.5,

        # ── Аугментации ──

        # Геометрические
        degrees=8.0,         
        translate=0.1,       
        scale=0.4,            
        shear=1.5,            
        perspective=0.0003,   
        fliplr=0.5,           
        flipud=0.0,           

        # Смешивание
        mixup=0.1,            
        erasing=0.1,         

        # Яркость (без изменения цвета)
        hsv_h=0.0,            
        hsv_s=0.0,            
        hsv_v=0.1,           

        # Отключено
        copy_paste=0.0,  
        mosaic=1.0,

        # ── Прочее ──
        workers=4,
        seed=seed,
        verbose=True,
        val=True,
        plots=True,
    )
    return model

In [ ]:
model = train_model(
    data_yaml=yaml_path,
    model_name="yolov9c.pt",
    epochs=30,
    imgsz=640,
    batch=12,
    project="train_v9c",
    seed=42,
)

In [ ]:
# ============================================================
# 8. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ ОБУЧЕНИЯ
# ============================================================

from IPython.display import display, Image as IPImage
from pathlib import Path

# Папка с результатами обучения
train_dir = RUNS_DIR / "train_v9c"

# ── Все графики, которые YOLO генерирует автоматически ──
plot_files = [
    # Кривые обучения (loss, метрики по эпохам)
    "results.png",

    # Матрица ошибок
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",

    # Кривые метрик детекции
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
    "PR_curve.png",

    # Примеры предсказаний на val
    "val_batch0_pred.png",
    "val_batch1_pred.png",
    "val_batch2_pred.png",

    # Ground truth для сравнения
    "val_batch0_labels.png",
    "val_batch1_labels.png",
    "val_batch2_labels.png",

    # Примеры из train (с аугментациями)
    "train_batch0.jpg",
    "train_batch1.jpg",
    "train_batch2.jpg",

    # Распределение labels
    "labels.jpg",
    "labels_correlogram.jpg",
]

print("=" * 60)
print("РЕЗУЛЬТАТЫ ОБУЧЕНИЯ")
print("=" * 60)

for fname in plot_files:
    fpath = train_dir / fname
    if fpath.exists():
        print(f"\n{'─' * 60}")
        print(f"📊 {fname}")
        print(f"{'─' * 60}")
        display(IPImage(filename=str(fpath), width=900))
    else:
        print(f"  ⚠️  {fname} — не найден")

# ── Вывод лучших метрик из results.csv ──
results_csv = train_dir / "results.csv"
if results_csv.exists():
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]

    print("\n" + "=" * 60)
    print("ЛУЧШИЕ МЕТРИКИ (по mAP50-95)")
    print("=" * 60)

    # Находим эпоху с лучшим mAP50-95
    map_col = [c for c in df.columns if "mAP50-95" in c and "val" not in c.lower()]
    if not map_col:
        map_col = [c for c in df.columns if "mAP50-95" in c]

    if map_col:
        best_idx = df[map_col[0]].idxmax()
        best_row = df.iloc[best_idx]

        print(f"\n  Лучшая эпоха: {int(best_row['epoch']) + 1}")
        print()

        # Метрики
        metric_cols = [c for c in df.columns if any(
            m in c for m in ["precision", "recall", "mAP50", "mAP50-95"]
        )]
        for col in metric_cols:
            print(f"  {col:<30} {best_row[col]:.4f}")

        # Loss
        print()
        loss_cols = [c for c in df.columns if "loss" in c.lower()]
        for col in loss_cols:
            print(f"  {col:<30} {best_row[col]:.4f}")

    # Последняя эпоха
    print(f"\n{'─' * 60}")
    print(f"ПОСЛЕДНЯЯ ЭПОХА ({int(df.iloc[-1]['epoch']) + 1})")
    print(f"{'─' * 60}")

    last = df.iloc[-1]
    for col in df.columns:
        if col != "epoch":
            print(f"  {col:<30} {last[col]:.4f}")

    # Общая статистика
    print(f"\n{'─' * 60}")
    print("ОБЩАЯ СТАТИСТИКА")
    print(f"{'─' * 60}")
    print(f"  Всего эпох обучено:     {len(df)}")
    print(f"  Лучшая эпоха:           {int(df[map_col[0]].idxmax()) + 1}")

    if map_col:
        print(f"  Лучший mAP50-95:        {df[map_col[0]].max():.4f}")
        mAP50_col = [c for c in df.columns if "mAP50" in c and "95" not in c]
        if mAP50_col:
            print(f"  Лучший mAP50:            {df[mAP50_col[0]].max():.4f}")

# ── Информация о весах ──
print(f"\n{'=' * 60}")
print("СОХРАНЁННЫЕ ВЕСА")
print(f"{'=' * 60}")

weights_dir = train_dir / "weights"
if weights_dir.exists():
    for w in sorted(weights_dir.iterdir()):
        size_mb = w.stat().st_size / 1024 / 1024
        print(f"  {w.name:<20} {size_mb:.1f} MB")
else:
    print("  ⚠️  Папка weights не найдена")

print(f"\n{'=' * 60}")

In [ ]:
# ============================================================
# 9. ЗАГРУЗКА ЛУЧШИХ ВЕСОВ
# ============================================================

best_pt = RUNS_DIR / "train_v9c" / "weights" / "best.pt"
last_pt = RUNS_DIR / "train_v9c" / "weights" / "last.pt"
weights = best_pt if best_pt.exists() else last_pt
model   = YOLO(str(weights))

In [ ]:
# ============================================================
# 10. ИНФЕРЕНС НА ТЕСТОВЫХ ИЗОБРАЖЕНИЯХ  (imgsz=640)
# ============================================================

results = model.predict(
    source=str(TEST_IMAGES),
    imgsz=640,
    conf=0.20,
    iou=0.50,
    max_det=100,
    augment=True,
    save_txt=True,
    save_conf=True,
    save=False,
    stream=True,
    verbose=False,
    project=str(RUNS_DIR),
    name="predict",
    exist_ok=True,
)

n_images = sum(1 for _ in results)
print(f"Инференс: {n_images} изображений")

In [ ]:
# ============================================================
# 11. ФОРМИРОВАНИЕ SUBMISSION
# ============================================================

def build_submission(sample_csv, preds_dir,
                     output_csv="submission.csv", target_class=1):
    sol = pd.read_csv(sample_csv)
    labels_dir = Path(preds_dir) / "labels"

    rows, total_boxes = [], 0

    for idx, row in sol.iterrows():
        img_name = str(row["image_name"])
        pred_file = labels_dir / f"{Path(img_name).stem}.txt"

        boxes = []
        if pred_file.exists():
            text = pred_file.read_text().strip()
            if text:
                for line in text.splitlines():
                    parts = line.split()
                    if len(parts) < 6:
                        continue
                    cls = int(float(parts[0]))
                    if cls != target_class:
                        continue
                    xc, yc, w, h = map(float, parts[1:5])
                    conf = float(parts[5])
                    boxes.append([xc, yc, w, h, conf])

        total_boxes += len(boxes)
        rows.append({
            "id":         idx,
            "image_name": img_name,
            "boxes":      json.dumps(boxes, separators=(",", ":")),
        })

    sub = pd.DataFrame(rows)
    sub.to_csv(output_csv, index=False)

    imgs_with_det = sum(1 for r in rows if r["boxes"] != "[]")
    print(f"{output_csv}: {len(sub)} imgs, "
          f"{imgs_with_det} w/ detections, {total_boxes} boxes total")
    return sub